In [ ]:
!pip install pandas langchain faiss-cpu sentence-transformers

# CSV → Vector DB (semantic) → HS Object → TOON Prompt → Chatbot Response


In [ ]:
import pandas as pd

df = pd.read_excel("/content/drive/MyDrive/HS_Code_Rag/HSCodeMaster v3.2.xlsx", dtype=str)
pd.set_option('display.max_colwidth', None)
df.head()

,OldHSCode,NewHSCode,LongDescAr,LongDescEn,Unit,Consignment,StatisticalQtyUnit,IS_Vehicle,CDM_Priority,DutyPercentage,AltDutyPercentage,ServiceCharge,Exemption,SpecificDuty,SpecificDutyQty,VGNMat
0,01012110,010121100001,خيول اصيلة الانسال من اصل عربي - ذكور خيول من اصل عربي للأنسال,Live pure-bred breeding horses of Arabian breed - Of Arab breed males,القيمةVALUE,Normal,Kilograms,No,Medium,0,0,0,NaN,NaN,NaN,New Sub
1,01012110,010121100002,خيول اصيلة الانسال من اصل عربي - إناث خيول من أصل عربي للأنسال,Live pure-bred breeding horses of Arabian breed - Of Arab breed females,القيمةVALUE,Normal,Kilograms,No,Medium,0,0,0,NaN,NaN,NaN,New Sub
2,01012190,010121900001,خيول اصيلة الانسال ما عدا الخيول التي من اصل عربي - ذكور خيول من أصل غير عربي للأنسال,Other live pure-bred breeding horses other than of Arabian breed. - Of non-Arab breed males,القيمةVALUE,Normal,Pieces/Units,No,Medium,0,0,0,NaN,NaN,NaN,New Sub
3,01012190,010121900002,خيول اصيلة الانسال ما عدا الخيول التي من اصل عربي - إناث خيول من أصل غير عربي للأنسال,Other live pure-bred breeding horses other than of Arabian breed. - Of non-Arab breed females,القيمةVALUE,Normal,Pieces/Units,No,Medium,0,0,0,NaN,NaN,NaN,New Sub
4,01012910,010129100001,خيول رياضه من أصل غير عربي للرياضة,Live sport horses other than pur- bred breeding animals - Males of Arab breed for sport,القيمةVALUE,Normal,Pieces/Units,No,Medium,0,0,0,NaN,NaN,NaN,New Sub


In [ ]:
df.drop(["Unit","Consignment","IS_Vehicle","CDM_Priority","VGNMat"], axis = 1, inplace = True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#df.drop(["OldHSCode"], axis = 1, inplace = True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13449 entries, 0 to 13448
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   OldHSCode           13449 non-null  object
 1   NewHSCode           13449 non-null  object
 2   LongDescAr          13449 non-null  object
 3   LongDescEn          13449 non-null  object
 4   StatisticalQtyUnit  13449 non-null  object
 5   DutyPercentage      13449 non-null  object
 6   AltDutyPercentage   13449 non-null  object
 7   ServiceCharge       13449 non-null  object
 8   Exemption           7106 non-null   object
 9   SpecificDuty        11074 non-null  object
 10  SpecificDutyQty     10834 non-null  object
dtypes: object(11)
memory usage: 1.1+ MB


In [ ]:
df = df.fillna("0")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13449 entries, 0 to 13448
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   OldHSCode           13449 non-null  object
 1   NewHSCode           13449 non-null  object
 2   LongDescAr          13449 non-null  object
 3   LongDescEn          13449 non-null  object
 4   StatisticalQtyUnit  13449 non-null  object
 5   DutyPercentage      13449 non-null  object
 6   AltDutyPercentage   13449 non-null  object
 7   ServiceCharge       13449 non-null  object
 8   Exemption           13449 non-null  object
 9   SpecificDuty        13449 non-null  object
 10  SpecificDutyQty     13449 non-null  object
dtypes: object(11)
memory usage: 1.1+ MB


In [ ]:
df.rename(columns={'NewHSCode': 'final_hs_code'}, inplace=True)

In [ ]:
import re

def normalize_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"[-–—]", " ", text)   # normalize hyphens/dashes
    text = re.sub(r"\.", " ", text)       # remove dots
    text = re.sub(r"\s+", " ", text)     # collapse multiple spaces
    return text.strip()


In [ ]:
df["LongDescEn"] = df["LongDescEn"].apply(normalize_text)

In [ ]:
df["LongDescEn"].head()

,LongDescEn
0,live pure bred breeding horses of arabian breed of arab breed males
1,live pure bred breeding horses of arabian breed of arab breed females
2,other live pure bred breeding horses other than of arabian breed of non arab breed males
3,other live pure bred breeding horses other than of arabian breed of non arab breed females
4,live sport horses other than pur bred breeding animals males of arab breed for sport


In [ ]:
import re

def normalize_arabic_text(text):
    if not isinstance(text, str):
        return ""

    # Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)

    # Remove tashkeel (diacritics)
    text = re.sub(r"[\u064B-\u0652]", "", text)

    # Remove punctuation and symbols
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


In [ ]:
df["LongDescAr"] = df["LongDescAr"].apply(normalize_arabic_text)

In [ ]:
df["LongDescAr"].head()

,LongDescAr
0,خيول اصيله الانسال من اصل عربي ذكور خيول من اصل عربي للانسال
1,خيول اصيله الانسال من اصل عربي اناث خيول من اصل عربي للانسال
2,خيول اصيله الانسال ما عدا الخيول التي من اصل عربي ذكور خيول من اصل غير عربي للانسال
3,خيول اصيله الانسال ما عدا الخيول التي من اصل عربي اناث خيول من اصل غير عربي للانسال
4,خيول رياضه من اصل غير عربي للرياضه


In [ ]:
#df.to_csv("/content/drive/MyDrive/HS_Code_Rag/hs_code_processed_ar_en.csv", index=False)

# Converting Each Row → Document

In [ ]:
from langchain_core.documents import Document


documents = []



for _, row in df.iterrows():

    metadata = {
        "FinalHSCode": row["final_hs_code"],
        "OldHSCode": row["OldHSCode"],
        "StatisticalQtyUnit": row["StatisticalQtyUnit"],
        "DutyPercentage": row["DutyPercentage"],
        "AltDutyPercentage": row["AltDutyPercentage"],
        "ServiceCharge": row["ServiceCharge"],
        "Exemption": row["Exemption"],
        "SpecificDuty": row["SpecificDuty"],
        "SpecificDutyQty": row["SpecificDutyQty"],
    }

    # English document
    documents.append(
        Document(
            page_content=f"Product description: {row['LongDescEn']}",
            metadata={**metadata, "lang": "en"}
        )
    )

    # Arabic document
    documents.append(
        Document(
            page_content=f"Product description: {row['LongDescAr']}",
            metadata={**metadata, "lang": "ar"}
        )
    )


In [ ]:
for doc in documents[:10]:
    print(doc)
    print("-" * 80)

page_content='Product description: live pure bred breeding horses of arabian breed of arab breed males' metadata={'FinalHSCode': '010121100001', 'OldHSCode': '01012110', 'StatisticalQtyUnit': 'Kilograms', 'DutyPercentage': '0', 'AltDutyPercentage': '0', 'ServiceCharge': '0', 'Exemption': '0', 'SpecificDuty': '0', 'SpecificDutyQty': '0', 'lang': 'en'}
--------------------------------------------------------------------------------
page_content='Product description: خيول اصيله الانسال من اصل عربي ذكور خيول من اصل عربي للانسال' metadata={'FinalHSCode': '010121100001', 'OldHSCode': '01012110', 'StatisticalQtyUnit': 'Kilograms', 'DutyPercentage': '0', 'AltDutyPercentage': '0', 'ServiceCharge': '0', 'Exemption': '0', 'SpecificDuty': '0', 'SpecificDutyQty': '0', 'lang': 'ar'}
--------------------------------------------------------------------------------
page_content='Product description: live pure bred breeding horses of arabian breed of arab breed females' metadata={'FinalHSCode': '0101211

In [ ]:
pip install -U langchain langchain-community

**Creating Embeddings**

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load local embeddings model
embedding_model = HuggingFaceEmbeddings(model_name="omarelshehy/arabic-english-sts-matryoshka-v2.0")

/tmp/ipython-input-1070318103.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="omarelshehy/arabic-english-sts-matryoshka-v2.0")


# Building vector store

In [ ]:
"""from langchain_community.vectorstores import FAISS
FAISS_PATH = "/content/drive/MyDrive/HS_Code_Rag/faiss_hscode_index_ar_en"

vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embedding_model
)
vectorstore.save_local(FAISS_PATH)"""

'from langchain_community.vectorstores import FAISS\nFAISS_PATH = "/content/drive/MyDrive/HS_Rag/faiss_hscode_index_ar_en"\n\nvectorstore = FAISS.from_documents(\n    documents=documents,\n    embedding=embedding_model\n)\nvectorstore.save_local(FAISS_PATH)'

**Creating retriever**

In [ ]:
from langchain_community.vectorstores import FAISS

# Loading saved vectorstore
FAISS_PATH = "/content/drive/MyDrive/HS_Code_Rag/faiss_hscode_index_ar_en"
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embedding_model,
    allow_dangerous_deserialization=True
)


**TOON Format**

In [ ]:
def format_context_toon(docs):
    if not isinstance(docs, list):
        docs = [docs]

    fields = [

        "final_hs_code",
        "description",
        "duty_fee",
        #"OldHSCode",
        #"statistical_qty_unit",
        # "alternate_duty_percent",
        # "service_charge",
        # "exemption",
        # "specific_duty",
        # "specific_duty_qty",
    ]

    header = f"hs_records[{len(docs)}]{{{','.join(fields)}}}:"

    rows = []
    for doc in docs:
        row = [

            doc.metadata.get("FinalHSCode"),
            doc.page_content.replace("Product description:", "").strip(),
            doc.metadata.get("DutyPercentage"),
            #doc.metadata.get("OldHSCode"),
            #doc.metadata.get("StatisticalQtyUnit"),
            # doc.metadata.get("AltDutyPercentage"),
            # doc.metadata.get("ServiceCharge"),
            # doc.metadata.get("Exemption"),
            # doc.metadata.get("SpecificDuty"),
            # doc.metadata.get("SpecificDutyQty"),
        ]
        rows.append(",".join("" if v is None else str(v) for v in row))


    return header + "\n" + "\n".join(rows)


In [ ]:
TOON_PROMPT = """
You are a customs declaration verifier.
You are given one authoritative HS record as JSON.
Verify the incoming declaration against the HS record and report each field separately:

HS code:
- If it matches, say: Declaration HS code is correct.
- If not, say: For this description, the declared HS code is incorrect.
  Then provide the correct HS code from the HS record.
- Never use or mention old_hs_code in the response.

Description:
- If it matches, say: Declaration description is matching.

Duty fee:
- If it matches, say: Declaration duty fee is correct.
- If not, say: The declared duty fee is incorrect.
  Then provide the correct duty value from the HS record.

Risk rule:
- If HS code, description, and duty fee are all correct → "The declaration is non risky."
- Otherwise → "The declaration is risky."


Rules:
- Do NOT infer missing data or add explanations.
- Do NOT modify numeric values.
- If a value is not present in the HS record, say: Not available in the HS dataset.
- Output only the verification results.

HS_RECORD:
{hs_record}

INCOMING_DECLARATION:
{incoming_declaration}

OUTPUT RESPONSE (use this exact format):

Risk: <result>
HS code: <result>
Description: <result>
Duty fee: <result>
"""


In [ ]:
import requests

OLLAMA_URL = "http://185.216.21.192:11434/api/generate"
MODEL_NAME = "llama3.1:8b"

def llm(prompt: str) -> str:
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False
        },
        timeout=120
    )

    response.raise_for_status()
    return response.json()["response"]


In [ ]:
hs_codes_set = set(df['final_hs_code'].unique())

def verify_hs_code(hs_code):
    return hs_code in hs_codes_set

def detect_language(text):
    return "ar" if re.search(r"[\u0600-\u06FF]", text) else "en"

In [ ]:
import json
def hscode_verification_toon(incoming_declaration):

    # STEP 1: Get raw description
    query = incoming_declaration.get("description", "")

    # STEP 2: Detect language
    lang = detect_language(query)
    print("language:", lang)

    # STEP 3: Normalize based on language
    if lang == "ar":
        cleaned_query = normalize_arabic_text(query)
    else:
        cleaned_query = normalize_text(query)

    # STEP 4: Update declaration with cleaned description
    incoming_declaration["description"] = cleaned_query


    retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"lang": lang}
    })

    # STEP 5: Retrieval using cleaned description
    result = retriever.invoke(cleaned_query)

    if not result:
        return "No matching HS code found for the given description."

    docs = result if isinstance(result, list) else [result]

    hs_record = format_context_toon(docs)

    # STEP 6: Prompt
    prompt = TOON_PROMPT.format(
        hs_record=json.dumps(hs_record, indent=2),
        incoming_declaration=incoming_declaration
    )

    #print("PROMPT---")
    #print(prompt)

    return llm(prompt)

In [ ]:
from IPython.display import HTML, display


# MAIN EXECUTION FLOW

incoming_declaration = {
    "hs_code": "010121100002",
    "description": "Live pure-bred breeding  horses of Arabian  breed - Of Arab breed females",
    "duty_fee": "0",
}

#incoming_description = normalize_text(incoming_declaration["description"])

# STEP 1: HS code verification
hs_code = verify_hs_code(incoming_declaration.get("hs_code"))
print(hs_code)

if hs_code:
    # STEP 2: Continue RAG pipeline only if HS code is valid
    response = hscode_verification_toon(incoming_declaration)
else:
    response = "Invalid HS Code. Not available in the HS dataset."

# STEP 3: Display response
display(
    HTML(
        f"<div style='white-space: pre-wrap; "
        f"max-width: 100%; "
        f"font-family: monospace; "
        f"font-size: 14px;'>"
        f"{response}"
        f"</div>"
    )
)


True
language: en


In [ ]:
from IPython.display import HTML, display


# MAIN EXECUTION FLOW

incoming_declaration = {
    "hs_code": "010121900001",
    "description": "خيول رياضه ما عدا  الخيول التي من اصل عربي   - إناث خيول من أصل عربي للرياضة",
    "duty_fee": "5",
}

#incoming_description = normalize_text(incoming_declaration["description"])

# STEP 1: HS code verification
hs_code = verify_hs_code(incoming_declaration.get("hs_code"))
print(hs_code)

if hs_code:
    # STEP 2: Continue RAG pipeline only if HS code is valid
    response = hscode_verification_toon(incoming_declaration)
else:
    response = "Invalid HS Code. Not available in the HS dataset."

# STEP 3: Display response
display(
    HTML(
        f"<div style='white-space: pre-wrap; "
        f"max-width: 100%; "
        f"font-family: monospace; "
        f"font-size: 14px;'>"
        f"{response}"
        f"</div>"
    )
)


True
language: ar


**Streamlit Part**

In [ ]:
!pip install -U langchain langchain-community pandas langchain faiss-cpu sentence-transformers streamlit pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstal

In [ ]:
%%writefile app.py
import streamlit as st
from IPython.display import HTML
import pandas as pd
import re
import requests
import json
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


df = pd.read_csv("/content/drive/MyDrive/HS_Code_Rag/hs_code_processed_ar_en.csv", dtype=str)

hs_codes_set = set(df['final_hs_code'].unique())

def verify_hs_code(hs_code):
    return hs_code in hs_codes_set

def detect_language(text):
    return "ar" if re.search(r"[\u0600-\u06FF]", text) else "en"

def normalize_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"[-–—]", " ", text)   # normalize hyphens/dashes
    text = re.sub(r"\.", " ", text)       # remove dots
    text = re.sub(r"\s+", " ", text)     # collapse multiple spaces
    return text.strip()

def normalize_arabic_text(text):
    if not isinstance(text, str):
        return ""

    # Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)

    # Remove tashkeel (diacritics)
    text = re.sub(r"[\u064B-\u0652]", "", text)

    # Remove punctuation and symbols
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Load local embeddings model
embedding_model = HuggingFaceEmbeddings(model_name="omarelshehy/arabic-english-sts-matryoshka-v2.0")


# Loading saved vectorstore
FAISS_PATH = "/content/drive/MyDrive/HS_Code_Rag/faiss_hscode_index_ar_en"
vectorstore = FAISS.load_local(
    FAISS_PATH,
    embedding_model,
    allow_dangerous_deserialization=True
)


def hscode_verification_toon(incoming_declaration):

    # STEP 1: Get raw description
    query = incoming_declaration.get("description", "")

    # STEP 2: Detect language
    lang = detect_language(query)
    print("language:", lang)

    # STEP 3: Normalize based on language
    if lang == "ar":
        cleaned_query = normalize_arabic_text(query)
    else:
        cleaned_query = normalize_text(query)

    # STEP 4: Update declaration with cleaned description
    incoming_declaration["description"] = cleaned_query


    retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"lang": lang}
    })

    # STEP 5: Retrieval using cleaned description
    result = retriever.invoke(cleaned_query)

    if not result:
        return "No matching HS code found for the given description."

    docs = result if isinstance(result, list) else [result]

    hs_record = format_context_toon(docs)

    # STEP 6: Prompt
    prompt = TOON_PROMPT.format(
        hs_record=json.dumps(hs_record, indent=2),
        incoming_declaration=incoming_declaration
    )

    print("PROMPT---")
    print(prompt)

    return llm(prompt)


def format_context_toon(docs):
    if not isinstance(docs, list):
        docs = [docs]

    fields = [

        "final_hs_code",
        #"OldHSCode",
        "description",
        "statistical_qty_unit",
        "duty_fee",
        # "alternate_duty_percent",
        # "service_charge",
        # "exemption",
        # "specific_duty",
        # "specific_duty_qty",
    ]

    header = f"hs_records[{len(docs)}]{{{','.join(fields)}}}:"

    rows = []
    for doc in docs:
        row = [

            doc.metadata.get("FinalHSCode"),
            #doc.metadata.get("OldHSCode"),
            doc.page_content.replace("Product description:", "").strip(),
            doc.metadata.get("StatisticalQtyUnit"),
            doc.metadata.get("DutyPercentage"),
            # doc.metadata.get("AltDutyPercentage"),
            # doc.metadata.get("ServiceCharge"),
            # doc.metadata.get("Exemption"),
            # doc.metadata.get("SpecificDuty"),
            # doc.metadata.get("SpecificDutyQty"),
        ]
        rows.append(",".join("" if v is None else str(v) for v in row))

    return header + "\n" + "\n".join(rows)


TOON_PROMPT = """
You are a customs declaration verifier.
You are given one authoritative HS record as JSON.
Verify the incoming declaration against the HS record and report each field separately:

HS code:
- If it matches, say: Declaration HS code is correct.
- If not, say: For this description, the declared HS code is incorrect.
  Then provide the correct HS code from the HS record.
- Never use or mention old_hs_code in the response.

Description:
- If it matches, say: Declaration description is matching.

Duty fee:
- If it matches, say: Declaration duty fee is correct.
- If not, say: The declared duty fee is incorrect.
  Then provide the correct duty fee from the HS record.

Risk rule:
- If HS code, description, and duty fee are all correct → "The declaration is non risky."
- Otherwise → "The declaration is risky."

Rules:
- Do NOT infer missing data or add explanations.
- Do NOT modify numeric values.
- If a value is not present in the HS record, say: Not available in the HS dataset.
- Output only the verification results.

HS_RECORD:
{hs_record}

INCOMING_DECLARATION:
{incoming_declaration}

OUTPUT RESPONSE (use this exact format):
Here are the verification results:
Risk: <result>
HS code: <result>
Description: <result>
Duty fee: <result>
"""




OLLAMA_URL = "http://185.216.21.192:11434/api/generate"
MODEL_NAME = "llama3.1:8b"

def llm(prompt: str) -> str:
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False
        },
        timeout=120
    )

    response.raise_for_status()
    return response.json()["response"]


# Streamlit Page Config
st.set_page_config(
    page_title="HS Code Declaration Verifier"
    #layout="wide"
)

st.title("Customs Declaration Verification")
st.caption("HS Code | Description | Duty Fee Verification (Arabic & English)")

# User Inputs
with st.form("declaration_form"):
    hs_code = st.text_input(
        "HS Code",
        placeholder="e.g. 010121900001"
    )

    description = st.text_area(
        "Goods Description (Arabic / English)",
        height=120,
        placeholder="Enter goods description"
    )

    duty_fee = st.text_input(
        "Declared Duty Fee",
        placeholder="e.g. 5"
    )

    submitted = st.form_submit_button("Verify Declaration")

# Processing
if submitted:

    if not description.strip():
        st.error("Description is required")
    else:
        incoming_declaration = {
            "hs_code": hs_code.strip(),
            "description": description.strip(),
            "duty_fee": duty_fee.strip()
        }

        st.subheader("Verification Result")

        # STEP 1: HS code validation (same logic as notebook)
        hs_valid = verify_hs_code(incoming_declaration.get("hs_code"))

        if hs_valid:
            with st.spinner("Running HS verification..."):
                response = hscode_verification_toon(incoming_declaration)
        else:
            response = "Invalid HS Code. Not available in the HS dataset."


        st.markdown(
            f"""
            <div style="
                font-family: 'Segoe UI', Arial, sans-serif;
                font-size: 14px;
                line-height: 1.3;
                border: 1px solid #e0e0e0;
                padding: 16px;
                border-radius: 8px;
                background-color: #ffffff;
                box-shadow: 0 1px 2px rgba(0,0,0,0.05);
            ">
                {response}
            </div>
            """,
            unsafe_allow_html=True
        )



Overwriting app.py


In [ ]:
!ngrok config add-authtoken 34rym51lPBarWBiLEqQaJFwJC6Y_Q4MhJeo2286pvmCEE2co

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok
!streamlit run app.py &>/dev/null&
url = ngrok.connect(8501)
print("🌐 App running at:", url)

🌐 App running at: NgrokTunnel: "https://alina-uninstalled-aleida.ngrok-free.dev" -> "http://localhost:8501"


# Evaluation

Arabic- Evaluation

In [ ]:
import pandas as pd
import json
import os

# # Paths
# CSV_PATH = "/content/drive/MyDrive/HS_Code_Rag/hs_code_processed_ar_en.csv"
OUTPUT_PATH = "/content/drive/MyDrive/HS_Code_Rag/hs_code_validation_ar.json"

# # Load HS dataset
# df = pd.read_csv(CSV_PATH)

# Keep only required columns for Arabic validation
df = df[["LongDescAr", "final_hs_code", "DutyPercentage"]].dropna()

# # OPTIONAL: remove duplicates
# df = df.drop_duplicates(subset=["LongDescAr", "final_hs_code"])

# OPTIONAL: sample N rows for validation (adjust SAMPLE_SIZE if needed)
SAMPLE_SIZE = 500
df_sample = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)

# Build validation records
validation_data = []
for _, row in df_sample.iterrows():
    validation_data.append({
        "query": row["LongDescAr"].strip(),
        "true_final_hs_code": str(row["final_hs_code"]),
        "true_duty_fee": str(row["DutyPercentage"])
    })

# Save to JSON
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(validation_data, f, indent=2, ensure_ascii=False)

print(f"Arabic validation dataset created with {len(validation_data)} samples")
print(f"Saved at: {OUTPUT_PATH}")


Arabic validation dataset created with 500 samples
Saved at: /content/drive/MyDrive/HS_Code_Rag/hs_code_validation_ar.json


In [ ]:
import json

VAL_PATH = "/content/drive/MyDrive/HS_Code_Rag/hs_code_validation_ar.json"

with open(VAL_PATH, "r") as f:
    eval_data = json.load(f)


In [ ]:
import re

def extract_final_hs_code(text):
    match = re.search(r"\b\d{12}\b", text)
    return match.group(0) if match else None

def extract_duty_fee(text):
    """
    Extracts the duty fee from the LLM response text.
    Returns the duty fee as a string, or None if not found.
    """
    # Look for a number (integer or decimal) following "Duty" or "duty fee"
    match = re.search(r"(?:Duty fee|duty fee|duty|Fee)\s*[:\-]?\s*(\d+(\.\d+)?)", text, re.IGNORECASE)
    if match:
        return match.group(1)
    return None


**Approach 1**

In [ ]:
TOON_PROMPT = """You are a customs declaration verifier.

You are given multiple retrieved HS records as JSON.
Exactly ONE of these records is authoritative for the given incoming declaration.

Your task has TWO phases:

PHASE 1: HS RECORD SELECTION
- Compare the incoming declaration description with each retrieved HS record.
- Select the ONE HS record that best matches the incoming declaration description.
- Do not explain the selection process.
- Do not mention scores or reasoning.
- The selected HS record becomes the authoritative HS record.
- Ignore all other retrieved records.

PHASE 2: VERIFICATION
Verify the incoming declaration ONLY against the selected authoritative HS record and report each field separately:

HS code:
- If it matches, say: Declaration HS code is correct.
- If not, say: For this description, the declared HS code is incorrect.
  Then provide the correct HS code from the authoritative HS record.
- Never use or mention old_hs_code.

Description:
- If it matches, say: Declaration description is matching.
- If not, say: The declared description is incorrect.
  Then provide the correct description from the authoritative HS record.

Duty fee:
- If it matches, say: Declaration duty fee is correct.
- If not, say: The declared duty fee is incorrect.
  Then provide the correct duty value from the authoritative HS record.

Rules:
- The HS record is final and must not be questioned.
- Do NOT infer missing data.
- Do NOT modify numeric values.
- Do NOT add explanations.
- If a value is not present in the HS record, say: Not available in the HS dataset.
- Output only what is requested below.

OUTPUT FORMAT (strict):

Retrieved Declaration:
hs_code: <value from selected HS record>
description: <value from selected HS record>
duty_fee: <value from selected HS record>

Verification Result:
<HS code verification>
<Description verification>
<Duty fee verification>

HS_RECORDS:
{hs_record}

INCOMING_DECLARATION:
{incoming_declaration}
"""

In [ ]:
import json

def hscode_verification_toon(incoming_declaration):
    """
    Accepts either:
    1) dict -> {"hs_code": ..., "description": ..., "duty_fee": ...}
    2) str  -> description only (used during evaluation)
    """

    # -------- STEP 0: Normalize input type --------
    if isinstance(incoming_declaration, str):
        incoming_declaration = {
            "hs_code": "",
            "description": incoming_declaration,
            "duty_fee": ""
        }

    if not isinstance(incoming_declaration, dict):
        raise ValueError("incoming_declaration must be dict or string")

    # Getting raw description
    query = incoming_declaration.get("description", "")

    # Detect language
    lang = detect_language(query)
    #print("language---:", lang)

    # Normalize text
    if lang == "ar":
        cleaned_query = normalize_arabic_text(query)
    else:
        cleaned_query = normalize_text(query)

    # Update declaration
    incoming_declaration["description"] = cleaned_query

    # Retriever
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": 2,
            "filter": {"lang": lang}
        }
    )

    #Retrieve
    docs = retriever.invoke(cleaned_query)

    if not docs:
        return "No matching HS code found for the given description."

    if not isinstance(docs, list):
        docs = [docs]

    # Format HS record
    hs_record = format_context_toon(docs)

    # Prompt
    prompt = TOON_PROMPT.format(
        hs_record=hs_record,
        incoming_declaration=incoming_declaration
    )

    #print("Prompt")
    #print(prompt)

    return llm(prompt)


In [ ]:
def evaluate_accuracy_ar_en(eval_data):
    total = len(eval_data)
    correct_hs = 0
    correct_duty = 0
    results = []

    for item in eval_data:
        incoming_declaration = {
            "hs_code": item.get("true_final_hs_code", ""),
            "description": item["query"],
            "duty_fee": item.get("true_duty_fee", "")
        }

        # HS code gate check (same as production)
        if verify_hs_code(incoming_declaration["hs_code"]):
            response = hscode_verification_toon(incoming_declaration)
        else:
            response = "Invalid HS Code. Not available in the HS dataset."


        #print("model response----:", response)

        predicted_hs = extract_final_hs_code(response)
        predicted_duty = extract_duty_fee(response)

        #hs_correct = predicted_hs == item["true_final_hs_code"]
        #duty_correct = predicted_duty == str(item["true_duty_fee"])


        hs_correct = (
            predicted_hs is not None
            and str(predicted_hs) == str(item["true_final_hs_code"])
        )

        duty_correct = (
            predicted_duty is not None
            and str(predicted_duty) == str(item["true_duty_fee"])
        )

        #print("predicted_hs---", str(predicted_hs))
        #print("true final hs code---", str(item["true_final_hs_code"]))



        #hs_correct = str(incoming_declaration["hs_code"]) == str(docs.metadata["FinalHSCode"])
        #duty_correct = float(incoming_declaration["duty_fee"]) == float(docs.metadata["DutyPercentage"])

        if hs_correct:
            correct_hs += 1
        if duty_correct:
            correct_duty += 1

        results.append({
            "description": item["query"],
            "true_hs_code": item["true_final_hs_code"],
            "predicted_hs_code": predicted_hs,
            "true_duty_fee": item["true_duty_fee"],
            "predicted_duty_fee": predicted_duty,
            "hs_correct": hs_correct,
            "duty_correct": duty_correct
        })
        import json

    for r in results[:5]:
        print(json.dumps(r, ensure_ascii=False, indent=2))



    return correct_hs / total, correct_duty / total, results


In [ ]:
accuracy_hs, accuracy_duty, detailed_results = evaluate_accuracy_ar_en(
    eval_data
)

print(f"HS Code Accuracy: {accuracy_hs:.2%}")
print(f"Duty Fee Accuracy: {accuracy_duty:.2%}")


predicted_hs--- 841460000000
true final hs code--- 841460000000
predicted_hs--- 950662000003
true final hs code--- 950662000002
predicted_hs--- 400911100000
true final hs code--- 400911100000
predicted_hs--- 480700000001
true final hs code--- 480700000001
predicted_hs--- 080410100000
true final hs code--- 080410100000
predicted_hs--- 290549100000
true final hs code--- 290549100000
predicted_hs--- 320414000000
true final hs code--- 320414000000
predicted_hs--- 440910900000
true final hs code--- 440910900000
predicted_hs--- 600536000000
true final hs code--- 600536000000
predicted_hs--- 251320300001
true final hs code--- 251320300002
predicted_hs--- 070999100000
true final hs code--- 070999100000
predicted_hs--- 843290000002
true final hs code--- 843290000002
predicted_hs--- 851432000000
true final hs code--- 851432000000
predicted_hs--- 284180000001
true final hs code--- 284180000001
predicted_hs--- 290389900001
true final hs code--- 290389900001
predicted_hs--- 580620000000
true final 

**English-Arabic Evaluation**

In [ ]:
import pandas as pd
import json

OUTPUT_PATH = "/content/drive/MyDrive/HS_Code_Rag/hs_code_validation_ar_en.json"

# Keep required columns
df = df[["LongDescAr", "LongDescEn", "final_hs_code", "DutyPercentage"]].dropna()

# Optional: remove duplicates per language
df = df.drop_duplicates(
    subset=["LongDescAr", "LongDescEn", "final_hs_code", "DutyPercentage"]
)

# SAMPLE SIZE
SAMPLE_SIZE_PER_LANG = 200

# Sample Arabic
df_ar = df.sample(
    n=min(SAMPLE_SIZE_PER_LANG, len(df)),
    random_state=42
)

# Sample English (different seed to avoid overlap bias)
df_en = df.sample(
    n=min(SAMPLE_SIZE_PER_LANG, len(df)),
    random_state=99
)

validation_data = []

#Build Arabic records
for _, row in df_ar.iterrows():
    validation_data.append({
        "query": row["LongDescAr"].strip(),
        "true_final_hs_code": str(row["final_hs_code"]),
        "true_duty_fee": str(row["DutyPercentage"])
    })

#Build English records
for _, row in df_en.iterrows():
    validation_data.append({
        "query": row["LongDescEn"].strip(),
        "true_final_hs_code": str(row["final_hs_code"]),
        "true_duty_fee": str(row["DutyPercentage"])
    })

# Save JSON
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(validation_data, f, indent=2, ensure_ascii=False)

print(f"Validation dataset created with {len(validation_data)} samples")
print(f"Arabic samples: {len(df_ar)}")
print(f"English samples: {len(df_en)}")
print(f"Saved at: {OUTPUT_PATH}")


Validation dataset created with 400 samples
Arabic samples: 200
English samples: 200
Saved at: /content/drive/MyDrive/HS_Code_Rag/hs_code_validation_ar_en.json


In [ ]:
accuracy_hs, accuracy_duty, detailed_results = evaluate_accuracy_ar_en(
    eval_data
)

print(f"HS Code Accuracy: {accuracy_hs:.2%}")
print(f"Duty Fee Accuracy: {accuracy_duty:.2%}")


{
  "description": "اجهزه اغطيه شافطه او مبدله للهواء، لايتجاوز مقاس اكبر جانب افقي لها سم، وان كانت مزوده بمنقيات هواء فلاتر",
  "true_hs_code": "841460000000",
  "predicted_hs_code": "841460000000",
  "true_duty_fee": "5",
  "predicted_duty_fee": "5",
  "hs_correct": true,
  "duty_correct": true
}
{
  "description": "كرات التنس العشبي قابله للنفخ كرات سله",
  "true_hs_code": "950662000002",
  "predicted_hs_code": "950662000002",
  "true_duty_fee": "5",
  "predicted_duty_fee": "5",
  "hs_correct": true,
  "duty_correct": true
}
{
  "description": "انابيب ومواسير وخراطيم من مطاط مبركن مهياه لمعدات النقل غير مقواه او متحده بطريقه اخري بمواد اخر، دون لوازم",
  "true_hs_code": "400911100000",
  "predicted_hs_code": "400911100000",
  "true_duty_fee": "5",
  "predicted_duty_fee": "5",
  "hs_correct": true,
  "duty_correct": true
}
{
  "description": "ورق وورق مقوي، مجمع طبقات مسطحه باللصق ، غير مطلي السطح ولا مشرب، وان كان مقوي من الداخل، لفات او صفايح ورق مجمع مصنع بلصق طبقات مسطحه من الور